In [44]:
import cv2
import numpy as np
# from google.colab.patches import cv2_imshow

# Sharpening using smooth filters




> We've seen that smooth filters are used to blur and remove info from the image.
<br>
In this part your task is to do the opposite ! Use them to actually make the image sharper



- hint: $$
g_{\text{sharp}} = f + \gamma \left( f - h_{\text{blur}} * f \right) $$


In [45]:

def sharpen_image(image:np.ndarray, kernel_size=(7, 7), gamma=2):
  gaussian_image = cv2.GaussianBlur(image,kernel_size,0)
  
  
  diff_image = image.astype(int) - gaussian_image.astype(int)
  
  
  sharpened_detail = gamma * diff_image

  res = sharpened_detail + image.astype(int)
  res.clip()
  res = np.clip(res, 0, 255).astype(np.uint8)
  

  return res,gaussian_image

image = cv2.imread('pictures/tiger.jpg')
if image is not None:
  sharpened_img, blur = sharpen_image(image)
  cv2.imshow('original',image)
  cv2.imshow('sharp',sharpened_img)
  cv2.imshow('blur',blur)
  cv2.waitKey(0)
  cv2.destroyAllWindows()
else:
  print("Error: Could not load image.")
if np.array_equal(image,sharpened_img):
  print('yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy')

# Morphological Edge Detection

- Extract clean object boundaries by subtracting eroded image from its dilated version


In [46]:
# Read the image
img = cv2.imread("pictures/bacteria.jpg", cv2.IMREAD_GRAYSCALE)
# For black and white image:
_, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# TODO: Morphological edge detection (dilate and then erode and then ...)
kernel_open_size = (3,3)
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,kernel_open_size)
res_dial = cv2.dilate(binary,kernel_open)
res_erosion = cv2.erode(binary,kernel_open)
edge_img = cv2.subtract(res_dial,res_erosion)
# TODO: clean edges with morphological closing
kernel_close_size = (3,3)
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,kernel_close_size)
res = cv2.morphologyEx(edge_img,cv2.MORPH_CLOSE,kernel_close)
# # TODO: Show results
cv2.imshow('result',res)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Count bacteria

bonus:
- find a way to calculate the kernel size automatically (based on overlaps or cell sizes for example)
- reconstruct cells by dilation after counting them

In [47]:
# Load image (grayscale)
img = cv2.imread("pictures/bacteria.jpg", cv2.IMREAD_GRAYSCALE)

# Otsu thresholding
# For black and white image:
_, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# Flip the colors (for morphology)
binary = cv2.bitwise_not(binary)
contours = cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

kernel_erosion_size = (9,9)
kernel_erosion = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,kernel_erosion_size)

eroded_binary = cv2.erode(binary,kernel_erosion,iterations=5)


# TODO: Connected component analysis
analysis = cv2.connectedComponentsWithStats(eroded_binary)
num_labels, labels, stats, centroids = analysis
# TODO: print the number of components
print(f'number of components: {num_labels - 1}')
# TODO: Draw results (Boxes)
res = eroded_binary.copy()
res = cv2.cvtColor(res,cv2.COLOR_GRAY2BGR)
reconstruct_kernel_size = (15,15)
reconstruct_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,reconstruct_kernel_size)
reconstructed_image = cv2.dilate(res,reconstruct_kernel,iterations=4)
for i in range(1,num_labels):
    x = stats[i,cv2.CC_STAT_LEFT]
    y = stats[i,cv2.CC_STAT_TOP]
    width =  stats[i,cv2.CC_STAT_WIDTH]
    height = stats[i,cv2.CC_STAT_HEIGHT]
    x_center,y_center = centroids[i]
    x_center,y_center = int(x_center),int(y_center)
    top_left = (x,y)
    bottom_right = (x + width,y + height)
    cv2.rectangle(res,top_left,bottom_right,color=(255,0,255),thickness=1)
    cv2.circle(res,(x_center,y_center),radius=2,color=(0,255,0),thickness=cv2.FILLED)
    cv2.putText(res,f'Component {i}',(x-30,y-10),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,0,255),1)
# TODO: Display
cv2.imshow('grayscale', img)
cv2.imshow('binary',binary)
cv2.imshow('eroded',eroded_binary)
cv2.imshow('result',res)
cv2.imshow('reconstructed',reconstructed_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


number of components: 26


# Optional



> Learn as much as you want about CNNs and ask mentors (CNNs will be taught in future sessions)



# Sources:
- Computer Vision: Algorithms and Applications
- Dr. Karimi videos
- Google
- My own knowledge